# Start here: inspect the ECG apnea study

Read the [PDF](../reports/final-study-report.pdf) or [final report](../docs/final-study-report.md). This notebook checks the included source data and curated artifact hashes, summarizes RQ1/RQ2, and replays one saved logistic regression model on its own held-out fold. **No fitting, threshold selection or downloads occur.**

This is exploratory retrospective research, with unverified subject identities and prior cohort exposure. Models are cross-validation artifacts, not clinical deployment models. The source dataset and derived data retain the attribution terms in `data/README.md`.

Use the project environment: `uv sync --locked`, then `uv run jupyter lab`. Detailed executed evidence snapshots are in `notebooks/executed/`.

In [1]:
import hashlib
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
R = ROOT / "data/processed/portfolio-v2/plan001-real-20261108/full/real-20261109-v1/results"
manifest = json.loads((R / "manifest.json").read_text())
original = json.loads((ROOT / "provenance/original-results-manifest.json").read_text())
assert manifest["curation"]["original_artifact_count"] == 140
assert len(manifest["artifacts"]) == 35
for name, expected in manifest["artifacts"].items():
    assert expected == original["artifacts"][name]
    assert hashlib.sha256((R / name).read_bytes()).hexdigest() == expected
models = list((R / "models").glob("*"))
assert len(models) == 25
source = json.loads((ROOT / "data/processed/phase1/manifest.json").read_text())
for name, expected in source["source_sha256"].items():
    assert (
        hashlib.sha256((ROOT / "data/raw/apnea-ecg-1.0.0" / name).read_bytes()).hexdigest()
        == expected
    )
print("Verified: 34 source recordings, 35 curated exports including 25 primary outer-fold models.")

Verified: 34 source recordings, 35 curated exports including 25 primary outer-fold models.


## Results at a glance

AP is average precision, not accuracy. Mean fold AP is the primary result; pooled metrics and descriptive fold SD are separate quantities. The constant-score mean fold AP is 0.395. The final report explains threshold failures and conditional uncertainty, including why a mean ranking alone does not establish every pairwise difference.

In [2]:
metrics = pd.read_csv(R / "metrics.csv")
primary = metrics.query("analysis == 'tuned' and seed == 67 and level == 'fold_mean'")
display(
    primary[["model", "average_precision", "fold_ap_std"]].sort_values(
        "average_precision", ascending=False
    )
)
fixed = metrics.query("analysis == 'ablation' and level == 'fold_mean'").pivot(
    index="model", columns="feature_set", values="average_precision"
)
display(fixed.round(3))
trials = pd.read_parquet(R / "trial_results.parquet")
assert len(trials) == 600 and trials.groupby(["model", "outer_fold"]).size().eq(24).all()
print("Accounted for all 24 candidates × five pipelines × five outer folds.")

,model,average_precision,fold_ap_std
1494,logistic_regression,0.646309,0.136802
1429,linear_svm,0.603273,0.170608
1754,xgboost,0.540138,0.136159
1559,random_forest,0.501570,0.098237
1234,cnn,0.474205,0.101343


feature_set,A_F1,B_F2,C_no_SDRR,D_no_RMSSD,E_no_pRR50
model,,,,,
logistic_regression,0.401,0.650,0.357,0.521,0.668
random_forest,0.400,0.507,0.355,0.489,0.519
xgboost,0.390,0.531,0.363,0.503,0.545


Accounted for all 24 candidates × five pipelines × five outer folds.


## Replay a saved model on its held-out fold

This loads a checksum-verified pickle produced by this study. Only load trusted serialized models. The LR pipeline contains its fitted imputation/scaling; raw feature values are supplied in the recorded order. IDs and labels are not predictors. This is a reproduction check of existing held-out predictions, not new evaluation or a full-cohort deployed model.

In [3]:
features = pd.read_parquet(ROOT / "data/processed/phase2-grouped/features.parquet")
predictions = pd.read_parquet(R / "outer_predictions.parquet")
held = predictions.query(
    "analysis == 'tuned' and model == 'logistic_regression' and seed == 67 and outer_fold == 0"
)
columns = ["mean_rr_ms", "median_rr_ms", "mean_hr_bpm", "sdrr_ms", "rmssd_ms", "prr50_pct"]
joined = held[["record_id", "minute", "label", "score"]].merge(
    features[["record_id", "minute", "label", *columns]],
    on=["record_id", "minute"],
    suffixes=("", "_feature"),
    validate="one_to_one",
)
assert len(joined) == len(held) and joined.label.equals(joined.label_feature)
model_path = R / "models/outer-0__tuned__logistic_regression__seed-67__outer.pkl"
with model_path.open("rb") as stream:
    model = pickle.load(stream)
scores = model.predict_proba(joined[columns].to_numpy())[
    :, int(np.flatnonzero(model.classes_ == 1)[0])
]
np.testing.assert_allclose(scores, joined.score.to_numpy(), atol=1e-12, rtol=0)
print(
    f"Replayed {len(scores):,} held-out scores; "
    f"max absolute error: {np.max(np.abs(scores - joined.score)):.3g}"
)

Replayed 3,381 held-out scores; max absolute error: 3.89e-16


## Further inspection

- [Notebook 09](09_nested_results.ipynb): fold/seed results, ablations and threshold diagnostics.
- [Notebook 10](10_cnn_implementation_audit.ipynb): exact CNN input reconstruction and model replay.
- [Notebook 11](11_cnn_curves_and_raw_signals.ipynb): selected training curves and original source signals.
- [Notebook 12](12_final_report.ipynb): check tables and regenerate the illustrated report.

Notebooks 10/11 and the raw-signal figure in 12 need the optional prepared-waveform release. Download and verify it with `python tools/fetch_prepared_waveforms.py`. See [REPRODUCING.md](../REPRODUCING.md). XGBoost must run in its documented isolated process without PyTorch. Do not change thresholds or select improved settings using these exposed outer results.